# 📚 만화 번역기 — Colab 실행 노트북

**런타임 → 런타임 유형 변경 → 하드웨어 가속기: `T4 GPU`** 선택 후 위에서부터 순서대로 실행하세요.

사전 준비: Colab 좌측 🔑(보안 비밀)에 `GEMINI_API_KEY`, `GITHUB_TOKEN` 등록.

In [ ]:
# 1) GPU 확인 (T4가 보여야 함)
!nvidia-smi

In [ ]:
# 2) 코드 클론 + 의존성 설치
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git clone https://{token}@github.com/auddpfkql-spec/Manga-translation-web-app-planning.git manga-translator
%cd manga-translator
!pip install -q -r requirements.txt

In [ ]:
# 3) ⚠️ CUDA torch 복구 (중요)
#    manga-ocr / paddleocr 등 의존성이 CPU 전용 torch 를 끌어와 Colab 의 CUDA torch 를
#    덮어쓰는 경우가 있다. CUDA(cu124) 빌드로 강제 재설치한다.
!pip install -q torch torchvision --force-reinstall --no-deps --index-url https://download.pytorch.org/whl/cu124
!ldconfig /usr/local/lib/python3.12/dist-packages/torch/lib

In [ ]:
# 4) CUDA 사용 가능 확인 (True 가 나와야 함)
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('❌ CUDA 불가 — 런타임을 T4 GPU 로 바꾸고 2~3 번을 다시 실행하세요.')

In [ ]:
# 5) 비밀키를 환경변수로 주입 (Colab userdata)
import os
from google.colab import userdata

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
try:
    os.environ['NGROK_AUTHTOKEN'] = userdata.get('NGROK_AUTHTOKEN')
except Exception:
    print('NGROK_AUTHTOKEN 없음 — 6-B(Colab 프록시) 셀을 사용하세요.')
print('환경변수 설정 완료')

In [ ]:
# 6-A) FastAPI 서버 + ngrok 터널 실행 (NGROK_AUTHTOKEN 필요)
import sys, os, threading
sys.path.insert(0, '/content/manga-translator')

import nest_asyncio, uvicorn
from pyngrok import ngrok

nest_asyncio.apply()
ngrok.set_auth_token(os.environ['NGROK_AUTHTOKEN'])

public_url = ngrok.connect(8000)
print('🌐 공개 URL:', public_url)

threading.Thread(
    target=lambda: uvicorn.run('app.main:app', host='0.0.0.0', port=8000, log_level='info'),
    daemon=True
).start()
print('서버 시작됨 — 위 공개 URL로 접속하세요.')

In [ ]:
# 6-B) (대안) ngrok 없이 Colab 프록시로 공개 — 토큰 불필요
import sys, os, threading
sys.path.insert(0, '/content/manga-translator')
os.chdir('/content/manga-translator')

import nest_asyncio, uvicorn
from google.colab.output import eval_js

nest_asyncio.apply()
print('🌐 접속 URL:', eval_js('google.colab.kernel.proxyPort(8000)'))

threading.Thread(
    target=lambda: uvicorn.run('app.main:app', host='0.0.0.0', port=8000, log_level='info'),
    daemon=True
).start()